# Laboratorio 4 — Cuaderno 14: análisis y conclusiones

**Ejercicio 10 de la Parte 2.**

Este cuaderno no produce resultados nuevos: reúne los de los seis anteriores para
responder a las tres preguntas del enunciado. ¿Sirve el modelo como herramienta de apoyo
al monitoreo? ¿Qué lo limita? ¿Qué haría falta para mejorarlo?

In [1]:
import sys
from pathlib import Path

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config, ml

validacion = joblib.load(config.MODELOS / "validacion_espacial_temporal.joblib")
generalizacion = joblib.load(config.MODELOS / "generalizacion_lagos.joblib")
interpretabilidad = joblib.load(config.MODELOS / "interpretabilidad.joblib")
predicciones = pd.read_parquet(config.DERIVADO / "predicciones_mapa.parquet")

print("Resultados de los cuadernos 9 a 13 cargados.")

Resultados de los cuadernos 9 a 13 cargados.


In [2]:
print("=" * 74)
print("RESUMEN DE TODOS LOS EXPERIMENTOS — XGBoost".center(74))
print("=" * 74)

pivote = validacion["pivote_f2"]
print("\n1. Estrategia de validación (F2 sobre todo el conjunto)\n")
print(pivote.to_string())

print("\n2. Generalización entre lagos (conjunto transferible, sin x ni y)\n")
tabla_gen = generalizacion["tabla"]
sub_gen = tabla_gen[tabla_gen["predictoras"] == "Transferible"]
print(sub_gen[["corrida", "Recall", "Precision", "F2", "PR-AUC"]].to_string(index=False))

print("\n3. Desempeño espacial fuera de muestra, por lago (cuaderno 13)\n")
for lago, sub in predicciones.groupby("lago"):
    vp = int((sub["tipo_error"] == 3).sum()); fn = int((sub["tipo_error"] == 2).sum())
    fp = int((sub["tipo_error"] == 1).sum())
    print(f"  {lago:<11} recall {vp/(vp+fn):.4f}   precisión {vp/(vp+fp):.4f}   "
          f"floración no detectada: {fn*0.0004:.2f} km²")

print("\n4. Variables más influyentes (SHAP medio absoluto)\n")
for var, valor in interpretabilidad["shap_medio"].head(5).items():
    print(f"  {var:<20}{valor:.3f}")

               RESUMEN DE TODOS LOS EXPERIMENTOS — XGBoost                

1. Estrategia de validación (F2 sobre todo el conjunto)

estrategia           Aleatoria  Espacial  Temporal  Δ espacial  Δ temporal
modelo                                                                    
Random Forest           0.9311    0.9186    0.6144     -0.0125     -0.3167
Regresión Logística     0.9229    0.9211    0.8372     -0.0018     -0.0857
XGBoost                 0.9439    0.9355    0.8176     -0.0084     -0.1263

2. Generalización entre lagos (conjunto transferible, sin x ni y)

              corrida  Recall  Precision     F2  PR-AUC
    Atitlan → Atitlan  0.9219     0.1779 0.5021  0.4166
Amatitlan → Amatitlan  0.9775     0.8759 0.9554  0.9870
  Atitlan → Amatitlan  0.5988     0.7212 0.6199  0.6735
  Amatitlan → Atitlan  0.8437     0.3947 0.6873  0.5455

3. Desempeño espacial fuera de muestra, por lago (cuaderno 13)

  Amatitlan   recall 0.9666   precisión 0.8602   floración no detectada: 0.58 k

## 10.1 ¿Tiene el modelo capacidad suficiente para apoyar el monitoreo?

**Sí, con dos condiciones y un lago de diferencia.** La respuesta corta esconde matices
que conviene desplegar, porque los resultados apuntan en direcciones distintas según qué
se le pida al modelo.

### Lo que sí sostiene la evidencia

**Detecta las floraciones que importan.** Sobre predicción fuera de muestra por bloques
espaciales, el modelo recupera el 96.7 % de los píxeles con alta presencia en Amatitlán.
Y el cuaderno 13 mostró que sus errores se concentran en la banda de 9.5 a 10.5 µg/L —el
borde mismo del umbral—, mientras que por encima de 13 µg/L falla en menos del 1 % de los
casos. Las floraciones que un gestor necesitaría atender de urgencia son precisamente las
que el modelo no deja pasar.

**Prioriza bien el esfuerzo de campo.** La escala de probabilidad del inciso 9.4 ordena de
forma monótona: 0.00 % de positivos reales en la categoría "muy baja", 3.86 % en "baja",
13.98 % en "alta" y 84.15 % en "muy alta". Un equipo que salga a muestrear las zonas
marcadas como muy alta acierta cinco de cada seis veces, frente al 1.18 % que obtendría
eligiendo al azar. Ese es el sentido operativo del trabajo: convertir 3.7 millones de
píxeles en una lista corta de sitios que vale la pena visitar.

**Aprendió física, no correlaciones espurias.** El cuaderno 12 lo confirma por tres
caminos independientes. El modelo se apoya en el realce del borde rojo (`B07`, SHAP 3.72),
en la razón azul/verde (3.39) —que es el principio de los algoritmos de color del océano
desde los años setenta— y en el contraste infrarrojo (`ndmi`) para separar biomasa de
sedimento. Las direcciones de los efectos son las que la óptica del agua predice. Un
modelo apoyado en variables físicamente sensatas resiste mejor las condiciones nuevas que
uno apoyado en casualidades del conjunto de entrenamiento.

**Se traslada bien en el espacio.** La validación por bloques de 1 km costó apenas 0.0084
de F2. El modelo puede predecir en zonas del lago donde nunca vio un píxel.

### Lo que la evidencia no sostiene

**No sirve igual en los dos lagos.** En Atitlán la precisión cae a 0.4694: **más de la
mitad de los avisos son falsos**. Con 1,324 píxeles positivos en 18 meses —el 0.039 % de
su superficie— no hay material para caracterizar el fenómeno allí. El cuaderno 11 lo dejó
crudo: entrenar en Amatitlán y evaluar en Atitlán (F2 = 0.6873) funciona mejor que
entrenar en el propio Atitlán (F2 = 0.5021). Para Atitlán esto es, por ahora, un
detector de anomalías con muchas falsas alarmas, no un sistema de monitoreo.

**No se traslada bien en el tiempo sin recalibrar.** La validación temporal costó 0.1263
de F2 a XGBoost y 0.3167 a Random Forest, cuyo recall se hundió de 0.9674 a 0.6101. El
diagnóstico del cuaderno 10 fue que el problema es de calibración y no de discriminación:
el ordenamiento de los píxeles aguanta —PR-AUC se mantiene en 0.9009— pero el umbral
aprendido en unas fechas corta mal en otras.

**No es una fuente independiente de verdad.** Este es el límite de fondo y merece decirse
sin adornos. La etiqueta no viene de un muestreo de campo: es un índice espectral
umbralado, calculado con B04 y B05. Que el modelo la reconstruya usando B07, B02 y B03
demuestra que la firma de una floración es redundante a lo ancho del espectro —un
resultado real sobre la óptica del agua— pero **no** demuestra que detectaría una
floración que el índice CyanoLakes no hubiera detectado. Si el índice se equivoca, el
modelo reproduce el error con la misma confianza.

### Conclusión operativa

El modelo **sí** puede usarse como herramienta de apoyo en Amatitlán, con dos condiciones:
recalibrar el umbral en cada escena nueva en vez de fijarlo una vez, y tratar sus salidas
como una **priorización de dónde muestrear**, nunca como un sustituto del muestreo. En
Atitlán haría falta acumular muchos más eventos positivos antes de darle el mismo uso.

Conviene además recordar qué se está reemplazando. La alternativa no es un sistema
perfecto: es no tener cobertura, o tenerla en unos pocos puntos de muestreo cada varias
semanas. Frente a eso, un mapa con 96.7 % de recall y una de cada seis alarmas falsas es
una mejora sustancial, aunque esté lejos de ser infalible.

## 10.2 Limitaciones encontradas

### Los datos y la variable respuesta

**No hay verdad de campo.** Es la limitación más severa y condiciona todas las demás. Sin
mediciones in situ de clorofila-a o de conteo celular no se puede saber si el índice
—y por tanto el modelo— acierta en términos absolutos. Todo el laboratorio mide
consistencia interna, no exactitud.

**La clorofila-a no es cianobacteria.** El pigmento lo tienen todas las algas. Las guías
de la OMS aplican el equivalente de 10 µg/L *cuando las cianobacterias dominan* la
comunidad, y aquí esa dominancia se asume sin comprobarla. El apoyo indirecto —la
correlación de 0.844 entre clorofila y material flotante en Amatitlán, que es la firma de
una nata— es razonable, pero en Atitlán esa correlación cae a 0.304 y la asunción se
debilita justo donde el modelo ya funciona peor.

**Binarizar destruye información.** El cuaderno 13 cuantificó el precio: 44.43 % de error
en la banda de 9.5 a 10.5 µg/L. Un píxel con 9.9 y otro con 10.1 tienen espectros
indistinguibles y etiquetas opuestas. Ninguna mejora del modelo puede arreglar eso, porque
la información física para separarlos no existe en la imagen.

**El desbalance extremo limita lo que se puede aprender.** 84 negativos por positivo a
nivel global y 2,536:1 en Atitlán. Obligó a submuestrear, a elegir el umbral por separado
y a usar PR-AUC en lugar de ROC-AUC.

### Resolución espacial y temporal

**20 metros por píxel.** Es la resolución nativa de las bandas de borde rojo y onda corta
de Sentinel-2, y no puede mejorarse sin cambiar de sensor. Las manchas incipientes menores
que un píxel se promedian con el agua limpia que las rodea y desaparecen. En Amatitlán,
con 15 km², la superficie útil son unos 36,700 píxeles: el lago entero cabe en una imagen
pequeña.

**Once fechas por lago en 18 meses.** Es muy poco para separar el ciclo estacional de la
tendencia. La validación temporal solo pudo formar 5 particiones sobre 21 fechas únicas, y
la varianza entre ellas fue enorme —rango de 0.258 en F2—. Con esa cantidad de fechas no
se puede afirmar que el deterioro de Amatitlán durante 2026 sea una tendencia y no una
coincidencia de escenas.

**La nubosidad decide qué fechas existen.** Guatemala está en el trópico y la temporada de
lluvias va de mayo a octubre. Las fechas disponibles no son una muestra aleatoria del
tiempo: son las que quedaron despejadas, y están sesgadas hacia la estación seca.
Cualquier floración asociada a las primeras lluvias —cuando el escurrimiento arrastra
nutrientes— está sistemáticamente subrepresentada. Además, L1C no trae capa de
clasificación de escena, así que el filtro de nube tuvo que armarse con la banda de cirrus
y una prueba de brillo, más tosco que el que ofrece L2A.

### Las diferencias entre lagos

Los dos lagos no son dos muestras del mismo problema. Difieren en área (123 contra
15 km²), en forma —distancia mediana a la orilla de 1,062 contra 206 m—, en estado trófico
(1.24 contra 6.29 µg/L de media) y sobre todo en la óptica del agua, porque el sedimento
del río Villalobos compite por la señal en Amatitlán y no en Atitlán. El cuaderno 12 lo
detectó de forma independiente: el modelo usa `ndmi` tres veces más en Amatitlán que en
Atitlán, precisamente para descontar ese sedimento.

Y con 272 veces más prevalencia en un lago que en el otro, cualquier modelo entrenado
sobre los dos juntos queda dominado por Amatitlán.

### La metodología de validación

**La división aleatoria del cuaderno 9 sobrestima.** Se conservó porque el enunciado la
pide y porque sirve de contraste, pero sus cifras son una cota superior.

**El umbral se elige con los mismos datos en los que luego se reporta la ventaja.** Se
mitigó con una validación interna que nunca toca el conjunto de prueba, pero la elección
de F2 como criterio es un juicio de valor —discutido en el inciso 5.3— y no un resultado.

**Cinco particiones temporales son pocas.** Con 21 fechas únicas no da para más sin dejar
particiones sin positivos, pero la varianza observada sugiere que el intervalo de confianza
real alrededor de esos números es ancho.

## 10.3 Qué datos adicionales mejorarían el modelo

Ordenados por lo que aportarían frente a lo que costarían.

### 1. Muestreo de campo coincidente con el paso del satélite

Es, con diferencia, lo más valioso, porque ataca la limitación de fondo: convertiría la
etiqueta de un índice espectral en una medición real. Bastaría con relativamente poco —
unas decenas de puntos por lago, tomados el mismo día del paso de Sentinel-2— para:

- **Validar el índice** y saber si el umbral de 10 µg/L se traduce donde debe.
- **Recalibrar el polinomio** de clorofila para estas aguas concretas. El del script
  CyanoLakes se calibró con datos simulados y no para lagos volcánicos tropicales; el
  cuaderno de la Parte I ya tuvo que corregir concentraciones negativas por debajo de
  NDCI = −0.0951.
- **Confirmar la dominancia de cianobacteria** con conteo celular, que es lo que la
  asunción del inciso 2.2 necesita y no tiene.

Sin esto, todo lo demás mejora un modelo cuya referencia sigue sin estar verificada.

### 2. Más fechas

Duplicar o triplicar la serie es lo más barato de todo: las imágenes ya existen y están
disponibles gratis. Sentinel-2 pasa cada cinco días con sus dos satélites, así que en 18
meses hay más de 100 oportunidades por lago, de las que se usaron 11. Con 40 o 50 fechas
por lago se podría:

- Formar particiones temporales de verdad, con validación hacia adelante en el tiempo, que
  es como se evalúa un sistema operativo.
- Separar estacionalidad de tendencia en el deterioro de Amatitlán.
- Recoger más eventos positivos en Atitlán, que es lo que ese lago necesita.

El límite práctico no es el dato sino el filtro de nubes: habría que aceptar escenas
parcialmente cubiertas y enmascarar por píxel en vez de descartar la fecha entera.

### 3. Variables meteorológicas e hidrológicas

Las floraciones responden a condiciones que la imagen no ve, y añadirlas daría al modelo
información genuinamente nueva en lugar de más formas de mirar el mismo espectro:

- **Temperatura del agua**, que Landsat 8/9 ofrece en el térmico y MODIS a diario. Las
  cianobacterias compiten mejor por encima de los 20 °C.
- **Precipitación acumulada en los días previos** — las fuentes que sugiere el enunciado
  dan climatología mensual, pero productos como CHIRPS o IMERG dan series diarias. El
  escurrimiento arrastra nutrientes: una floración suele venir días después de la lluvia.
- **Viento**, que determina si la nata se acumula en una orilla o se dispersa. Explicaría
  buena parte de la variación espacial que ahora el modelo no puede anticipar.
- **Caudal y carga de nutrientes del río Villalobos**, que es el origen conocido del
  problema en Amatitlán. Nitrógeno y fósforo son los que de verdad controlan el fenómeno,
  y ninguna banda espectral los mide.

Con estas variables el modelo podría pasar de **describir** la floración del día de la
imagen a **anticiparla**, que es un salto cualitativo en utilidad para la gestión.

### 4. Estructura temporal como predictor

Ninguna de las variables actuales mira al pasado del propio píxel. Añadir la clorofila de
la escena anterior, o la tendencia de las tres últimas, aportaría mucho: las floraciones
tienen inercia y persisten en los mismos focos, algo que la Parte I ya había documentado
en la mitad oriental de Amatitlán. Requiere series más densas —punto 2— para ser viable.

### 5. Más resolución espacial, con reservas

PlanetScope da 3 metros a diario, pero sus bandas no incluyen el borde rojo, que es la
señal principal del modelo. Sentinel-3 OLCI tiene bandas mucho mejores para clorofila
—incluida la de 709 nm— pero a 300 metros, lo que dejaría Amatitlán en unos 160 píxeles.
Ninguno domina a Sentinel-2 para este problema; lo razonable sería combinarlos, usando
OLCI para calibrar y Sentinel-2 para el detalle espacial.

---

## Cierre

El resultado más valioso de esta segunda parte no es el modelo sino lo que la validación
enseñó sobre él. Un experimento hecho sin cuidado —repartiendo píxeles al azar y dejando
B04 y B05 entre las predictoras— habría reportado un ROC-AUC de 1.0000 y cero falsos
negativos, y habría sido completamente vacío. Que la respuesta se derive de dos bandas
concretas, que los píxeles vecinos sean casi el mismo dato y que las fechas no sean
intercambiables son hechos del problema, no detalles técnicos, y solo aparecen si se los
busca.

Lo que queda tras quitar esos atajos es un modelo más modesto y mucho más creíble: capaz de
señalar dónde vale la pena ir a muestrear en Amatitlán, incapaz todavía de hacerlo con
fiabilidad en Atitlán, y dependiente de una referencia que nadie ha comprobado con un
frasco de agua.